# Experiment 0

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# Input Data
from google.colab import drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Sem 5/PMML/UCI_Credit_Card.csv'

df = pd.read_csv(file_path)


Mounted at /content/drive


In [ ]:
# Melihat Dimensi, Struktur & Tipe Dataset
print(f"\n DIMENSI DATASET ")
print(f"Jumlah Baris: {df.shape[0]}")
print(f"Jumlah Kolom: {df.shape[1]}")
print("\n STRUKTUR & TIPE DATA ")
print(df.info())


 DIMENSI DATASET 
Jumlah Baris: 30000
Jumlah Kolom: 25

 STRUKTUR & TIPE DATA 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT

In [ ]:
# Memeriksa Missing Value dan Duplikat
print("\n PEMERIKSAAN MISSING VALUE ")
total_null = df.isnull().sum().sum()
print(f"Total Missing Value di Seluruh Dataset: {total_null}")

print("\n PEMERIKSAAN DATA DUPLIKAT ")
total_duplicates = df.duplicated().sum()
print(f"Jumlah Baris Duplikat: {total_duplicates}")


 PEMERIKSAAN MISSING VALUE 
Total Missing Value di Seluruh Dataset: 0

 PEMERIKSAAN DATA DUPLIKAT 
Jumlah Baris Duplikat: 0


In [ ]:
# Memeriksa Anomali
print("\n PEMERIKSAAN NILAI ANOMALI ")
print("Distribusi awal EDUCATION:")
print(df['EDUCATION'].value_counts().sort_index())
print("\nDistribusi awal MARRIAGE:")
print(df['MARRIAGE'].value_counts().sort_index())

# Penanganan Anomali
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

print("\nDistribusi EDUCATION setelah diperbaiki:")
print(df['EDUCATION'].value_counts().sort_index())
print("\nDistribusi MARRIAGE setelah diperbaiki:")
print(df['MARRIAGE'].value_counts().sort_index())


 PEMERIKSAAN NILAI ANOMALI 
Distribusi awal EDUCATION:
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

Distribusi awal MARRIAGE:
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64

Distribusi EDUCATION setelah diperbaiki:
EDUCATION
1    10585
2    14030
3     4917
4      468
Name: count, dtype: int64

Distribusi MARRIAGE setelah diperbaiki:
MARRIAGE
1    13659
2    15964
3      377
Name: count, dtype: int64


In [ ]:
# Distribusi Variabel Target
print("\n DISTRIBUSI VARIABEL TARGET (default.payment.next.month) ")
target_counts = df['default.payment.next.month'].value_counts()
target_props = df['default.payment.next.month'].value_counts(normalize=True) * 100

print(f"Kelas 0 (Tidak Default) : {target_counts[0]} ({target_props[0]:.2f}%)")
print(f"Kelas 1 (Default)       : {target_counts[1]} ({target_props[1]:.2f}%)")


 DISTRIBUSI VARIABEL TARGET (default.payment.next.month) 
Kelas 0 (Tidak Default) : 23364 (77.88%)
Kelas 1 (Default)       : 6636 (22.12%)


In [ ]:
# Menentukan Predictor dan Target (Mengeluarkan Variabel ID)
# Mengeluarkan kolom 'ID' dan kolom target dari fitur prediktor (X)
X = df.drop(columns=['ID', 'default.payment.next.month'])
y = df['default.payment.next.month']

print(f"\n PENENTUAN PREDICTOR & TARGET ")
print(f"Fitur Prediktor (X) dimasukkan: {X.shape[1]} kolom (Kolom 'ID' berhasil dikeluarkan).")
print(f"Variabel Target (y): '{y.name}'")


 PENENTUAN PREDICTOR & TARGET 
Fitur Prediktor (X) dimasukkan: 23 kolom (Kolom 'ID' berhasil dikeluarkan).
Variabel Target (y): 'default.payment.next.month'


In [ ]:
# Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\n HASIL TRAIN-TEST SPLIT ")
print(f"Ukuran X_train (Training Set) : {X_train.shape}")
print(f"Ukuran X_test (Test Set)     : {X_test.shape}")
print(f"Ukuran y_train               : {y_train.shape}")
print(f"Ukuran y_test                : {y_test.shape}")


 HASIL TRAIN-TEST SPLIT 
Ukuran X_train (Training Set) : (24000, 23)
Ukuran X_test (Test Set)     : (6000, 23)
Ukuran y_train               : (24000,)
Ukuran y_test                : (6000,)


Eksperimen diawali dengan memuat dataset `UCI_Credit_Card.csv` yang terdiri dari 30.000 baris dan 25 kolom prediktor. Hasil pemeriksaan awal menunjukkan bahwa data bersih dari *missing value* maupun baris duplikat. Namun, ditemukan anomali nilai pada variabel kategorikal `EDUCATION` (nilai 0, 5, dan 6) serta `MARRIAGE` (nilai 0) yang tidak terdefinisi pada dokumentasi resmi. Nilai-nilai anomali tersebut kemudian digabungkan ke dalam kategori 'Others' (kategori 4 untuk `EDUCATION` dan kategori 3 untuk `MARRIAGE`) agar konsistensi data tetap terjaga.

Selanjutnya, variabel `'ID'` dikeluarkan dari himpunan fitur prediktor karena hanya berfungsi sebagai identifikasi unik yang tidak memiliki nilai ilmiah dalam memprediksi risiko default. Data kemudian dibagi menjadi 80% data latih (24.000 observasi) dan 20% data uji (6.000 observasi). Proses pemisahan ini menerapkan teknik *stratified splitting* pada variabel target `default.payment.next.month` untuk memastikan proporsi distribusi kelas mayoritas (77.88% non-default) dan kelas minoritas (22.12% default) seimbang secara konsisten di kedua subset data.

# Experiment 1

In [ ]:
# Baseline Model
baseline_model = LogisticRegression(max_iter=2000, random_state=42)

In [ ]:
# 5 Fold Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_f1_scores = cross_val_score(baseline_model, X_train, y_train, cv=cv, scoring='f1')
cv_f1_mean = cv_f1_scores.mean()


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

In [ ]:
# Melatih Model Pada Training Set
baseline_model.fit(X_train, y_train)
y_pred_exp1 = baseline_model.predict(X_test)


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# Menghitung dan Menampilkan Matriks Evaluasi
acc_exp1 = accuracy_score(y_test, y_pred_exp1)
prec_exp1 = precision_score(y_test, y_pred_exp1, zero_division=0)
rec_exp1 = recall_score(y_test, y_pred_exp1, zero_division=0)
f1_exp1 = f1_score(y_test, y_pred_exp1, zero_division=0)

exp1_results = pd.DataFrame({
    'Metric': ['CV F1-score (Mean)', 'Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1-score'],
    'Value': [cv_f1_mean, acc_exp1, prec_exp1, rec_exp1, f1_exp1]
})

print(" HASIL EVALUASI EXPERIMENT 1 (BASELINE MODEL) ")
print(exp1_results.to_string(index=False))

 HASIL EVALUASI EXPERIMENT 1 (BASELINE MODEL) 
            Metric    Value
CV F1-score (Mean) 0.362721
     Test Accuracy 0.808167
    Test Precision 0.688841
       Test Recall 0.241899
     Test F1-score 0.358059


Pada eksperimen pertama, dibangun model dasar (*baseline model*) menggunakan algoritma Logistic Regression tanpa menerapkan transformasi fitur maupun penanganan *class imbalance*. Evaluasi pada data latih menggunakan 5-Fold Stratified Cross-Validation menghasilkan rerata nilai CV F1-score sebesar 0.3627. Ketika diuji pada data uji (*test set*), model ini mencatatkan Akurasi sebesar 0.8082, Precision sebesar 0.6888, Recall sebesar 0.2419, dan F1-score sebesar 0.3581.

Hasil evaluasi menunjukkan adanya fenomena *accuracy paradox*. Meskipun akurasi model terlihat cukup tinggi di angka 80.82%, metrik Recall justru sangat rendah yaitu hanya 24.19%. Artinya, model gagal mendeteksi lebih dari 75% nasabah yang sebenarnya mengalami *default*. Kondisi ini terjadi karena algoritma Logistic Regression terdistorsi oleh perbedaan rentang skala variabel asli serta kecenderungan memprediksi kelas mayoritas akibat ketidakseimbangan proporsi data target.

# Experiment 2

In [ ]:
# Menentukan Kelompok Kolom Numerik dan Kategorikal
num_cols = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
            'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2',
            'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

In [ ]:
# Membuat ColumnTransformer untuk Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

In [ ]:
# Membangun Pipeline (Preprocessor + Logistic Regression)
exp2_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])


In [ ]:
#  Evaluasi 5-Fold Cross Validation pada Training Set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1_scores_exp2 = cross_val_score(exp2_pipeline, X_train, y_train, cv=cv, scoring='f1')
cv_f1_mean_exp2 = cv_f1_scores_exp2.mean()

In [ ]:
# Melatih Pipeline pada Training Set & Prediksi pada Test Set
exp2_pipeline.fit(X_train, y_train)
y_pred_exp2 = exp2_pipeline.predict(X_test)

In [ ]:
# Menghitung Metrik Evaluasi pada Test Set
acc_exp2 = accuracy_score(y_test, y_pred_exp2)
prec_exp2 = precision_score(y_test, y_pred_exp2, zero_division=0)
rec_exp2 = recall_score(y_test, y_pred_exp2, zero_division=0)
f1_exp2 = f1_score(y_test, y_pred_exp2, zero_division=0)

comparison_df = pd.DataFrame({
    'Metric': ['CV F1-score (Mean)', 'Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1-score'],
    'Experiment 1 (Baseline)': [cv_f1_mean, acc_exp1, prec_exp1, rec_exp1, f1_exp1],
    'Experiment 2 (Preprocessing)': [cv_f1_mean_exp2, acc_exp2, prec_exp2, rec_exp2, f1_exp2]
})

print(" HASIL EVALUASI & PERBANDINGAN EXPERIMENT 1 & EXPERIMENT 2 ")
print(comparison_df.to_string(index=False))

 HASIL EVALUASI & PERBANDINGAN EXPERIMENT 1 & EXPERIMENT 2 
            Metric  Experiment 1 (Baseline)  Experiment 2 (Preprocessing)
CV F1-score (Mean)                 0.362721                      0.468268
     Test Accuracy                 0.808167                      0.816833
    Test Precision                 0.688841                      0.661932
       Test Recall                 0.241899                      0.351168
     Test F1-score                 0.358059                      0.458887


Untuk mengatasi kelemahan pada model baseline, eksperimen kedua menerapkan *pipeline* pemrosesan awal (*preprocessing*) secara terstruktur. Fitur numerik sebanyak 14 kolom ditransformasikan menggunakan `StandardScaler` untuk menyamakan skala distribusi variabel kontinyu seperti jumlah limit kredit dan histori pembayaran. Sementara itu, 9 kolom fitur kategorikal dikonversi menggunakan `OneHotEncoder` guna menghindari asumsi tingkatan nilai. Seluruh tahapan ini dibungkus menggunakan `ColumnTransformer` dan `Pipeline` untuk memastikan tidak terjadi kebocoran data (*data leakage*) selama proses Cross-Validation.

Penerapan *preprocessing* memberikan dampak positif yang sangat signifikan terhadap performa model Logistic Regression. Nilai rerata CV F1-score meningkat drastis menjadi 0.4683 (naik +0.1056 dibanding baseline), sementara pada data uji diperoleh Test Akurasi sebesar 0.8168, Precision sebesar 0.6619, Recall sebesar 0.3512, dan Test F1-score sebesar 0.4589 (naik +0.1008 dibanding baseline). Hal ini membuktikan bahwa penyesuaian skala fitur sangat krusial bagi algoritma berbasis gradien untuk mencapai titik konvergensi yang optimal.

# Experiment 3

In [ ]:
# Definisi Model dalam Dictionary Pipeline (Menggunakan Preprocessor dari Exp 2)
models = {
    'Logistic Regression': Pipeline([
        ('prep', preprocessor),
        ('model', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Decision Tree': Pipeline([
        ('prep', preprocessor),
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('prep', preprocessor),
        ('model', RandomForestClassifier(random_state=42, class_weight='balanced'))
    ])
}

In [ ]:
# Menyiapkan Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# List tempat menyimpan hasil evaluasi
exp3_results = []

In [ ]:
# Loop Evaluasi Setiap Model
for name, model_pipeline in models.items():
    # Hitung CV F1-score pada Data Latih
    cv_scores = cross_val_score(model_pipeline, X_train, y_train, cv=cv, scoring='f1')
    cv_f1 = cv_scores.mean()

    # Fit model ke Training Set & Prediksi pada Test Set
    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)

    # Menghitung Metrik Evaluasi Test Set
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Simpan Hasil Metrik
    exp3_results.append({
        'Model': name,
        'CV F1-score': cv_f1,
        'Test Accuracy': acc,
        'Test Precision': prec,
        'Test Recall': rec,
        'Test F1-score': f1
    })

# Membuat Dataframe Perbandingan Model
exp3_df = pd.DataFrame(exp3_results)

print(" HASIL EVALUASI EXPERIMENT 3 (MODEL COMPARISON) ")
print(exp3_df.to_string(index=False))

 HASIL EVALUASI EXPERIMENT 3 (MODEL COMPARISON) 
              Model  CV F1-score  Test Accuracy  Test Precision  Test Recall  Test F1-score
Logistic Regression     0.468268       0.816833        0.661932     0.351168       0.458887
      Decision Tree     0.407141       0.718667        0.371530     0.393369       0.382138
      Random Forest     0.457017       0.813500        0.647309     0.344386       0.449582


Eksperimen ketiga bertujuan membandingkan performa tiga algoritma pembelajar yang berbeda, yaitu Logistic Regression, Decision Tree, dan Random Forest, di mana seluruh model dikombinasikan dengan *pipeline preprocessing* dari eksperimen sebelumnya. Evaluasi dilakukan secara adil menggunakan 5-Fold Cross-Validation pada data latih serta pengujian langsung pada data uji.

Hasil perbandingan menunjukkan bahwa Logistic Regression memimpin dengan CV F1-score sebesar 0.4683 dan Test F1-score sebesar 0.4589. Random Forest berada di posisi kedua dengan CV F1-score sebesar 0.4570 dan Test F1-score sebesar 0.4496, sedangkan Decision Tree memperoleh hasil terendah dengan CV F1-score sebesar 0.4071 dan Test F1-score sebesar 0.3821 akibat indikasi *overfitting*. Karena Logistic Regression terbukti memberikan performa paling tinggi dan paling stabil, model ini ditetapkan sebagai kandidat utama untuk dilanjutkan ke tahap pencarian hyperparameter terbaik.

# Experiment 4

In [ ]:
# Menyiapkan Pipeline dasar dengan Preprocessor dari Experiment 2
tuning_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
# Menentukan Ruang Parameter (Grid Search Space) yang Diuji
param_grid = {
    'model__C': [0.001, 0.01, 0.1, 1, 10],
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear'],
    'model__class_weight': [None, 'balanced']
}

In [ ]:
# Menyiapkan Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# Inisialisasi GridSearchCV dengan Scoring F1-score
grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

Proses Hyperparameter Tuning sedang berjalan...


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('prep',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['LIMIT_BAL',
                                                                          'AGE',
                                                                          'BILL_AMT1',
                                                                          'BILL_AMT2',
                                                                          'BILL_AMT3',
                                                                          'BILL_AMT4',
                                                                          'BILL_AMT5',
                                                                          'BILL_AMT6',
                                                                          'PAY_AMT1',
                                                                          'PAY_AMT2',
                                                                          'PAY_AMT3',
                                                                          'PAY_AMT4',
                                                                          'PAY_AMT5',
                                                                          'PAY_AMT6']),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['SEX',
                                                                          'EDUCATION',
                                                                          'MARRIAGE',
                                                                          'PAY_0',
                                                                          'PAY_2',
                                                                          'PAY_3',
                                                                          'PAY_4',
                                                                          'PAY_5',
                                                                          'PAY_6'])])),
                                       ('model',
                                        LogisticRegression(max_iter=1000,
                                                           random_state=42))]),
             n_jobs=-1,
             param_grid={'model__C': [0.001, 0.01, 0.1, 1, 10],
                         'model__class_weight': [None, 'balanced'],
                         'model__penalty': ['l1', 'l2'],
                         'model__solver': ['liblinear']},
             scoring='f1')

In [ ]:
# Mengambil Model Terbaik Hasil Tuning
best_logistic_model = grid_search.best_estimator_

In [ ]:
# Evaluasi Model Terbaik pada Data Uji (Test Set)
y_pred_exp4 = best_logistic_model.predict(X_test)

acc_exp4 = accuracy_score(y_test, y_pred_exp4)
prec_exp4 = precision_score(y_test, y_pred_exp4, zero_division=0)
rec_exp4 = recall_score(y_test, y_pred_exp4, zero_division=0)
f1_exp4 = f1_score(y_test, y_pred_exp4, zero_division=0)

In [ ]:
# Menampilkan Tabel Perbandingan Sebelum vs Setelah Tuning
comp_tuning_df = pd.DataFrame({
    'Metric': ['CV F1-score (Mean)', 'Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1-score'],
    'Sebelum Tuning (Exp 3 Default)': [0.468268, 0.816833, 0.661932, 0.351168, 0.458887],
    'Setelah Tuning (Exp 4 Best)': [grid_search.best_score_, acc_exp4, prec_exp4, rec_exp4, f1_exp4]
})

print("\n HYPERPARAMETER TUNING ")
print("Kombinasi Parameter Terbaik:", grid_search.best_params_)
print("\n HASIL EVALUASI SEBELUM & SETELAH TUNING ")
print(comp_tuning_df.to_string(index=False))


 HYPERPARAMETER TUNING 
Kombinasi Parameter Terbaik: {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__penalty': 'l1', 'model__solver': 'liblinear'}

 HASIL EVALUASI SEBELUM & SETELAH TUNING 
            Metric  Sebelum Tuning (Exp 3 Default)  Setelah Tuning (Exp 4 Best)
CV F1-score (Mean)                        0.468268                     0.534173
     Test Accuracy                        0.816833                     0.772667
    Test Precision                        0.661932                     0.487740
       Test Recall                        0.351168                     0.554635
     Test F1-score                        0.458887                     0.519041


Pada eksperimen keempat, dilakukan optimasi hyperparameter pada model Logistic Regression menggunakan metode `GridSearchCV` dengan skema 5-Fold Cross-Validation. Proses tuning mengeksplorasi variasi nilai parameter penalti, solver, koefisien regularisasi `C`, serta bobot kelas `class_weight`. Hasil optimasi menemukan bahwa kombinasi terbaik dicapai pada konfigurasi nilai `C=0.1`, jenis penalti L1 (`penalty='l1'`), solver `'liblinear'`, dan penanganan *class imbalance* berupa `class_weight='balanced'`.

Penggunaan konfigurasi terbaik ini menghasilkan peningkatan performa yang sangat dominan. Nilai rerata CV F1-score melonjak menjadi 0.5341, sedangkan pada data uji dicapai Test Akurasi sebesar 0.7727, Precision sebesar 0.4877, Recall sebesar 0.5546, dan Test F1-score sebesar 0.5190. Kehadiran pembobot kelas berhasil memicu kenaikan drastis pada metrik Recall dari 35.12% menjadi 55.46%, yang sangat penting dalam domain risiko kredit untuk menekan angka nasabah *gagal bayar* yang tidak terprediksi (*False Negative*).

# Experiment 5

In [ ]:
# Membangun Pipeline Advanced Model dengan Gradient Boosting
gb_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', GradientBoostingClassifier(random_state=42))
])

In [ ]:
# Menyiapkan 5-Fold Stratified Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
# Evaluasi Cross Validation F1-score pada Data Training
cv_scores_gb = cross_val_score(gb_pipeline, X_train, y_train, cv=cv, scoring='f1')
cv_f1_gb = cv_scores_gb.mean()

In [ ]:
# Melatih Model pada Seluruh Data Training & Prediksi Data Test
gb_pipeline.fit(X_train, y_train)
y_pred_gb = gb_pipeline.predict(X_test)

In [ ]:
# Menghitung Metrik Evaluasi Data Test
acc_gb = accuracy_score(y_test, y_pred_gb)
prec_gb = precision_score(y_test, y_pred_gb, zero_division=0)
rec_gb = recall_score(y_test, y_pred_gb, zero_division=0)
f1_gb = f1_score(y_test, y_pred_gb, zero_division=0)

# Menampilkan Perbandingan dengan Model Terbaik Experiment 4
comp_exp4_exp5 = pd.DataFrame({
    'Metric': ['CV F1-score (Mean)', 'Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1-score'],
    'Exp 4 (Logistic Reg Tuned)': [0.534173, 0.772667, 0.487740, 0.554635, 0.519041],
    'Exp 5 (Gradient Boosting Default)': [cv_f1_gb, acc_gb, prec_gb, rec_gb, f1_gb]
})

print(" HASIL EVALUASI EXPERIMENT 5 (ADVANCED MODEL) ")
print(comp_exp4_exp5.to_string(index=False))

 HASIL EVALUASI EXPERIMENT 5 (ADVANCED MODEL) 
            Metric  Exp 4 (Logistic Reg Tuned)  Exp 5 (Gradient Boosting Default)
CV F1-score (Mean)                    0.534173                           0.475368
     Test Accuracy                    0.772667                           0.818167
    Test Precision                    0.487740                           0.666667
       Test Recall                    0.554635                           0.355690
     Test F1-score                    0.519041                           0.463882


Eksperimen kelima menguji penerapan algoritma *ensemble* tingkat lanjut yaitu `GradientBoostingClassifier` default yang diintegrasikan ke dalam *pipeline preprocessing*. Pengujian dengan 5-Fold Cross-Validation menghasilkan nilai rerata CV F1-score sebesar 0.4754. Pada data uji, model ini memperoleh Test Akurasi sebesar 0.8182, Precision sebesar 0.6667, Recall sebesar 0.3557, dan Test F1-score sebesar 0.4639.

Meskipun model Gradient Boosting mampu melampaui kinerja model *baseline* dan model un-tuned lainnya, performa F1-score akhir (0.4639) masih berada di bawah Logistic Regression Tuned dari Experiment 4 (0.5190). Penyebab utamanya adalah algoritma Gradient Boosting versi standar belum menangani pembobotan kelas *imbalance*, sehingga kemampuan model dalam mendeteksi kelas minoritas (*Recall*) masih relatif rendah di kisaran 35.57%.

Berdasarkan seluruh rangkaian eksperimen yang telah dilakukan dari Experiment 1 hingga Experiment 5, terlihat perkembangan performa model yang signifikan. Model baseline Logistic Regression pada Experiment 1 hanya memperoleh Test F1-score sebesar 0.3581. Penerapan *preprocessing* pada Experiment 2 mampu menaikkan Test F1-score menjadi 0.4589. Pada pembandingan model di Experiment 3, Logistic Regression terbukti lebih unggul dibanding Decision Tree (Test F1: 0.3821) dan Random Forest (Test F1: 0.4496). Selanjutnya, penggunaan model lanjutan Gradient Boosting pada Experiment 5 menghasilkan Test F1-score sebesar 0.4639.

Dengan demikian, model **Logistic Regression (Tuned)** dari **Experiment 4** secara resmi dipilih sebagai **Final Predictive Model**. Model ini mencatatkan nilai F1-score tertinggi pada Cross-Validation (0.5342) maupun data uji (0.5190), serta memiliki nilai Recall tertinggi mencapai 55.46%. Pemilihan model ini sangat tepat bagi kebutuhan bisnis perbankan karena terbukti paling sensitif dan efektif dalam mengidentifikasi nasabah berisiko *default*, sekaligus meminimalkan potensi kerugian finansial perusahaan